# This file evaluates the generates responses using the selected metrics.

- Pass@1 for MBPP and HumanEval. Exact match for GSM8K.

In [1]:
ds_name = 'mbpp'
# ds_name = 'human_eval'
# ds_name = 'gsm8k'

import json
import os

# if in the root directory
if os.path.exists('README.md'):
	os.chdir('evaluation')
	os.chdir(f'{ds_name}-responses')

## GSM8K evaluation

In [ ]:
if ds_name != 'gsm8k':
	raise Exception('This script is only for GSM8K dataset responses.')

response_rows = []

for file_number in range(1, 100_000):
	filename = f'test_{file_number}.json'
	try:
		with open(filename) as file:
			# parse the JSON content
			data = json.load(file)
			if 'draft_1' not in data:
				data['draft_1'] = data['all_revisions'][0] \
					if data['all_revisions'] else data['draft_2']
			response_rows.append({
				'draft_1':  data.get('draft_1',  '').strip(),
				'draft_2':  data.get('draft_2',  '').strip(),
				'final_answer':   data.get('final_answer',   '').strip(),
				'correct_answer': data.get('correct_answer', '').strip(),
			})
	except:
		print(f'File {filename} not found or could not be read. Stopping.')
		break

total_rows = len(response_rows)
correct_count = 0
key_to_process = 'final_answer'  # evaluate 'draft_1', 'draft_2' or 'final_answer'

for index, row in enumerate(response_rows):
	if row[key_to_process] == row['correct_answer']:
		correct_count += 1
		continue

	# Answer is present after the last #### in the last line
	answer = row['correct_answer'].split('####')[-1].strip()
	# Check the presence of the answer in the final answer
	if answer not in row[key_to_process]:
		# Sometimes, 130000 and 130,000 are used interchangeably
		if answer.replace(',', '') not in row[key_to_process].replace(',', ''):
			print(f'Processing row {index + 1}/{total_rows}')
			print(f'Answer "{answer}" not found in {key_to_process}.')
			print('Given answer: \n', row[key_to_process], ' \n')
			print('Correct answer: \n', row['correct_answer'], ' \n')
			print('-' * 80)
			continue

	correct_count += 1

print(f'Total rows processed:', total_rows)
print(f'Correct answers:', correct_count)
print(f'Accuracy: {correct_count / total_rows * 100:.2f}%')

## Coding evaluations

In [2]:
import re

def get_code(response):
	if not response or not isinstance(response, str):
		print('Warning: Response is empty or not a string.')
		return None
	# Get the code based on tags
	codes = re.findall(f'```python\n(.*?)```',
					response, re.DOTALL | re.IGNORECASE)
	functions = [code.strip() for code in codes if 'def ' in code]
	if not functions:
		print('Warning: No function definitions found in the response.')
	return functions[0] if functions else None

### HumanEval

In [ ]:
# Using the official code for HumanEval: https://github.com/openai/human-eval

if ds_name != 'human_eval':
	print('This script is only for human_eval dataset.')
	exit(1)

from human_eval.data import write_jsonl
from human_eval.evaluation import evaluate_functional_correctness

# Load all responses
responses_data = []
for file_number in range(1, 100_000):
	filename = f'test_{file_number}.json'
	try:
		with open(filename) as file:
			# parse the JSON content
			data = json.load(file)
			if 'draft_1' not in data:
				data['draft_1'] = data['all_revisions'][0] \
					if data['all_revisions'] else data['draft_2']
			responses_data.append(data)
	except:
		print(f'File {filename} not found or could not be read. Stopping.')
		break

import pandas as pd
all_results = []

for col in ['draft_1', 'draft_2', 'final_answer']:

	samples = [
		dict(task_id=task['task_id'], completion=get_code(task[col]))
		for task in responses_data
	]

	sample_file = f'humaneval_samples_{col}.jsonl'
	write_jsonl(sample_file, samples)

	results = evaluate_functional_correctness(sample_file, ignore_incomplete=True)
	print(f'\n {col} results:', results, '\n')
	all_results.append({
		'col': col,
		'Pass@1 result': results['pass@1'],
	})

	try:
		os.remove(sample_file)
		os.remove(f'{sample_file}_results.jsonl')
	except:
		pass

print('HumanEval results:')
pd.DataFrame(all_results)

### MBPP

In [ ]:
if ds_name != 'mbpp':
	raise Exception('This script is only for MBPP dataset responses.')

# Load all responses
responses_data = []
for file_number in range(1, 100_000):
	filename = f'test_{file_number}.json'
	try:
		with open(filename) as file:
			# parse the JSON content
			data = json.load(file)
			responses_data.append(data)
	except:
		print(f'File {filename} not found or could not be read. Stopping.')
		break

# Loading test cases
samples = []
for row in responses_data:
	draft_1 = row.get('draft_1', None) or (row['all_revisions'][0] \
		if row['all_revisions'] else row['draft_2'])
	samples.append({
		'task_id': row['task_id'],
		# 'query': mbpp_row['prompt'].strip(),
		# 'code': mbpp_row['code'],
		'test_imports': row['test_imports'],
		'tests': row['test_list'],
		# 'draft_1': get_code(data['draft_1']),
		'draft_1_code': get_code(draft_1),
		# 'draft_2': draft_2,
		'draft_2_code': get_code(row['draft_2']),
		# 'final_answer': final_answer,
		'final_answer_code': get_code(row['final_answer']),
		'mbpp_code': row['mbpp_code'],
	})

def test_code(code: str, tests: list, silent=True) -> bool:
	code = code.strip()
	if not code:
		print('Warning: Empty code found, skipping tests.')
		return False
	try:
		exec(code)
		for test_case in tests:
			test_case = test_case.strip()
			if test_case:
				try:
					exec(test_case)
				except Exception as e:
					if not silent:
						print(f'Test failed: {test_case.strip()} with error: {e}')
					return False
	except Exception as e:
		return False

	return True  # All tests passed successfully

# Import the required modules
import sys
import collections
from collections import defaultdict
import heapq
import cmath
import math
from math import tan
import sys
from operator import itemgetter

# Testing the code using the test cases
correct_count_draft_1 = 0
correct_count_draft_2 = 0
correct_count_final_answer = 0
correct_count_mbpp_code = 0

for sample in samples:
	task_id = sample['task_id']  # Use task_id as index
	print(f'\nProcessing task {task_id}...')
	tests = sample['tests']

	# Import required modules
	for imp in sample['test_imports']:
		imp = imp.strip()
		if imp:
			try:
				exec(imp)
			except Exception as e:
				print(f'Import failed: {imp} with error: {e}')

	if test_code(sample['draft_1_code'], tests):
		correct_count_draft_1 += 1
	if test_code(sample['draft_2_code'], tests):
		correct_count_draft_2 += 1
	if test_code(sample['final_answer_code'], tests):
		correct_count_final_answer += 1
	if test_code(sample['mbpp_code'], tests):
		correct_count_mbpp_code += 1

total_count = len(samples)
print(f'Total tasks processed: {total_count}')
get_accuracy = lambda count: f'{count}/{total_count} ({count / total_count * 100:.2f}%)'
print(f'Correct draft 1:', get_accuracy(correct_count_draft_1))
print(f'Correct draft 2:', get_accuracy(correct_count_draft_2))
print(f'Correct final answer:', get_accuracy(correct_count_final_answer))

File test_2.json not found or could not be read. Stopping.

Processing task 11...
Total tasks processed: 1
Correct draft 1: 1/1 (100.00%)
Correct draft 2: 1/1 (100.00%)
Correct final answer: 1/1 (100.00%)
Correct MBPP code: 1/1 (100.00%)
